In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

ruta_dataset_1 = "MA_Train.csv"
ruta_dataset_2 = "MWB_Train.csv"

# Cargar los datasets en DataFrames de Pandas
ma_train = pd.read_csv(ruta_dataset_1)
mwb_train = pd.read_csv(ruta_dataset_2)

In [ ]:
def analizar_dataframe(df, nombre_df):

    print(f"\n==================================================")
    print(f"       GENERANDO ANÁLISIS PARA: {nombre_df}")
    print(f"==================================================")

    # ----------------------------------------------------
    # SELECCIÓN DE VARIABLES NUMÉRICAS
    # ----------------------------------------------------
    cols_num = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Eliminar IDs
    cols_num = [c for c in cols_num if 'id' not in c.lower()]

    if not cols_num:
        print(f"No se encontraron columnas numéricas en {nombre_df}.")
        return

    

    # ----------------------------------------------------
    # A. HISTOGRAMAS
    # ----------------------------------------------------
    num_cols = len(cols_num)
    fig_cols = 3
    fig_rows = (num_cols + fig_cols - 1) // fig_cols

    fig, axes = plt.subplots(
        fig_rows,
        fig_cols,
        figsize=(5 * fig_cols, 4 * fig_rows)
    )

    fig.suptitle(
        f'Histogramas de Variables Numéricas - {nombre_df}',
        fontsize=16,
        fontweight='bold'
    )

    axes_flat = axes.flatten() if num_cols > 1 else [axes]

    for idx, col in enumerate(cols_num):

        sns.histplot(
            df[col],
            kde=True,
            ax=axes_flat[idx],
            color='steelblue',
            bins=15
        )

        axes_flat[idx].set_title(
            f'Distribución: {col}',
            fontweight='bold'
        )

        axes_flat[idx].set_xlabel(col)
        axes_flat[idx].set_ylabel('Frecuencia')

    # Eliminar subplots vacíos
    for idx in range(num_cols, len(axes_flat)):
        fig.delaxes(axes_flat[idx])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # ----------------------------------------------------
    # B. BOXPLOTS
    # ----------------------------------------------------
    plt.figure(figsize=(14, len(cols_num) * 2))

    for i, col in enumerate(cols_num, 1):

        plt.subplot(
            (len(cols_num) + 1) // 2,
            2,
            i
        )

        sns.boxplot(
            x=df[col],
            color='skyblue'
        )

        plt.title(f'Boxplot de {col}')
        plt.xlabel(col)

    plt.tight_layout()
    plt.show()

    # ----------------------------------------------------
    # C. MATRIZ DE CORRELACIÓN
    # ----------------------------------------------------
    plt.figure(figsize=(10, 8))

    matriz_corr = df[cols_num].corr()

    sns.heatmap(
        matriz_corr,
        annot=True,
        fmt=".2f",
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        cbar_kws={"shrink": .8}
    )

    plt.title(
        f'Matriz de Correlación - {nombre_df}',
        fontsize=14,
        fontweight='bold',
        pad=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:

# ----------------------------------------------------
    # D. SCATTERPLOTS VS VARIABLE OBJETIVO
    # ----------------------------------------------------
def analizar_corr_target(df, nombre_df, target='Mental_Wellbeing_Score'):
    cols_num = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    # Eliminar IDs
    cols_num = [c for c in cols_num if 'id' not in c.lower()]
    
    if not cols_num:
        print(f"No se encontraron columnas numéricas en {nombre_df}.")
        return
    # Comprobar que existe la variable objetivo
    if target not in df.columns:
        print(f"No se encontró la variable objetivo '{target}' en {nombre_df}.")
        return
    
    # ----------------------------------------------------
    # CORRELACIONES CON LA VARIABLE OBJETIVO
    # ----------------------------------------------------
    print(f"\nCorrelación con {target}:")
    
    correlaciones_target = (
        df[cols_num]
        .corr()[target]
        .sort_values(ascending=False)
    )
    
    print(correlaciones_target)

    print(f"\n==================================================")
    print(f"   SCATTERPLOTS VS {target}: {nombre_df}")
    print(f"==================================================")

    # Excluir la propia variable objetivo
    scatter_cols = [
        c for c in cols_num
        if c != target
    ]

    if not scatter_cols:
        print("No hay variables numéricas para comparar con el objetivo.")
        return

    num_cols = len(scatter_cols)
    fig_cols = 3
    fig_rows = (num_cols + fig_cols - 1) // fig_cols

    fig, axes = plt.subplots(
        fig_rows,
        fig_cols,
        figsize=(5 * fig_cols, 4 * fig_rows)
    )

    fig.suptitle(
        f'Relación con {target} - {nombre_df}',
        fontsize=16,
        fontweight='bold'
    )

    axes_flat = axes.flatten() if num_cols > 1 else [axes]

    for idx, col in enumerate(scatter_cols):

        sns.regplot(
            data=df,
            x=col,
            y=target,
            ax=axes_flat[idx],
            scatter_kws={'alpha': 0.5},
            line_kws={'linewidth': 2}
        )

        axes_flat[idx].set_title(
            f'{col} vs {target}',
            fontweight='bold'
        )

        axes_flat[idx].set_xlabel(col)
        axes_flat[idx].set_ylabel(target)

    # Eliminar subplots vacíos
    for idx in range(num_cols, len(axes_flat)):
        fig.delaxes(axes_flat[idx])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
def analizar_categoricas(df, nombre_df, target="Mental_Wellbeing_Score"):

    print(f"\n==================================================")
    print(f"   ANÁLISIS DE VARIABLES CATEGÓRICAS: {nombre_df}")
    print(f"==================================================")

    # ----------------------------------------------------
    # SELECCIÓN DE VARIABLES CATEGÓRICAS
    # ----------------------------------------------------
    cols_cat = df.select_dtypes(
        include=['object', 'category', 'bool']
    ).columns.tolist()

    # Eliminar IDs
    cols_cat = [c for c in cols_cat if 'id' not in c.lower()]

    if not cols_cat:
        print("No se encontraron variables categóricas.")
        return

    if target not in df.columns:
        print(f"No se encontró la variable objetivo '{target}'.")
        return

    # ====================================================
    # A. DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS
    # ====================================================

    print("\nGenerando distribución de variables categóricas...")

    num_cols = len(cols_cat)
    fig_cols = 3
    fig_rows = (num_cols + fig_cols - 1) // fig_cols

    fig, axes = plt.subplots(
        fig_rows,
        fig_cols,
        figsize=(6 * fig_cols, 4.5 * fig_rows)
    )

    fig.suptitle(
        f'Distribución de Variables Categóricas - {nombre_df}',
        fontsize=16,
        fontweight='bold'
    )

    axes_flat = axes.flatten() if num_cols > 1 else [axes]

    for idx, col in enumerate(cols_cat):

        sns.countplot(
            data=df,
            x=col,
            ax=axes_flat[idx]
        )

        axes_flat[idx].set_title(
            f'Distribución de {col}',
            fontweight='bold'
        )

        axes_flat[idx].set_xlabel(col)
        axes_flat[idx].set_ylabel('Frecuencia')

        axes_flat[idx].tick_params(axis='x', rotation=45)

    # Eliminar subplots vacíos
    for idx in range(num_cols, len(axes_flat)):
        fig.delaxes(axes_flat[idx])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # ====================================================
    # B. VARIABLES CATEGÓRICAS VS TARGET
    # ====================================================

    print(f"\nGenerando boxplots de las variables categóricas vs {target}...")

    fig, axes = plt.subplots(
        fig_rows,
        fig_cols,
        figsize=(6 * fig_cols, 4.5 * fig_rows)
    )

    fig.suptitle(
        f'{target} según Variables Categóricas - {nombre_df}',
        fontsize=16,
        fontweight='bold'
    )

    axes_flat = axes.flatten() if num_cols > 1 else [axes]

    for idx, col in enumerate(cols_cat):

        sns.boxplot(
            data=df,
            x=col,
            y=target,
            ax=axes_flat[idx]
        )

        axes_flat[idx].set_title(
            f'{target} según {col}',
            fontweight='bold'
        )

        axes_flat[idx].set_xlabel(col)
        axes_flat[idx].set_ylabel(target)

        axes_flat[idx].tick_params(
            axis='x',
            rotation=45
        )

    for idx in range(num_cols, len(axes_flat)):
        fig.delaxes(axes_flat[idx])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
analizar_dataframe(mwb_train,"Dataset MWB (Mental Wellbeing) Train")
analizar_corr_target(mwb_train,"Dataset MWB (Mental Wellbeing) Train")
analizar_categoricas(mwb_train,"Dataset MWB (Mental Wellbeing) Train")

In [ ]:
analizar_dataframe(ma_train,"Dataset mental addiction Train")
analizar_corr_target(ma_train,"Dataset mental addiction Train",target = "GAD_7_Score")
analizar_corr_target(ma_train,"Dataset mental addiction Train",target = "PHQ_9_Score")
analizar_categoricas(ma_train,"Dataset mental addiction Train", target = "GAD_7_Score")
analizar_categoricas(ma_train,"Dataset mental addiction Train", target = "PHQ_9_Score")